# 자동화 코드가 갑자기 멈추는 진짜 이유
## — 데코레이터 · *args · **kwargs 제대로 이해하기

> **이 노트북의 목표**
> - 함수를 값처럼 다루는 개념 이해하기
> - 데코레이터가 왜 필요한지 스스로 발견하기
> - `*args`, `**kwargs`의 묶기/풀기 원리 완전 이해
> - 자동화 코드에서 어떻게 쓰이는지 실무 연결

---

## Part 1. 함수도 변수처럼 다룰 수 있다

파이썬에서 함수는 특별한 존재가 아니다.
숫자나 문자열처럼 변수에 담고, 인자로 넘길 수 있다.

In [ ]:
def hello():
    print("안녕")

# 함수를 변수에 담기
a = hello       # 실행이 아니라 함수 자체를 담음
print(type(a))  # <class 'function'>

a()             # 이때 실행됨

> **핵심 구분**
> - `hello` → 함수 그 자체 (실행 X)
> - `hello()` → 실행

In [ ]:
# 함수를 다른 함수에 넘기기
def run(func):
    func()

run(hello)  # hello를 인자로 전달

---

## Part 2. 데코레이터 — 필요에서 시작하기

먼저 문제를 보자.

In [ ]:
# 중복이 너무 많은 코드
def task_a():
    print("[로그] 실행 시작")
    print("A 작업")
    print("[로그] 실행 완료")

def task_b():
    print("[로그] 실행 시작")
    print("B 작업")
    print("[로그] 실행 완료")

# 함수가 100개면? 로그 코드도 100번 복붙?

In [ ]:
# 해결 아이디어: 함수를 받아서 기능을 붙이자
def add_log(func):
    def wrapper():
        print("[로그] 실행 시작")
        func()
        print("[로그] 실행 완료")
    return wrapper  # 새로운 함수를 반환

def task_a():
    print("A 작업")

# 함수를 바꿔치기
task_a = add_log(task_a)
task_a()

> **무슨 일이 일어났나?**
> 1. `add_log(task_a)` 호출 → `func = task_a` 저장
> 2. `wrapper` 함수 반환 → `task_a`가 `wrapper`로 교체됨
> 3. `task_a()` 호출 → 실제로는 `wrapper()` 실행

In [ ]:
# @decorator 문법 — 위 코드와 완전히 동일
@add_log
def task_b():
    print("B 작업")

# 내부적으로: task_b = add_log(task_b)
task_b()

---

## Part 3. `*args` — 여러 위치 인자 묶기/풀기

In [ ]:
# 기본: 위치 인자
def add(a, b):
    print(a, b)

add(1, 2)

In [ ]:
# *args: 여러 개를 묶어서 받기
def test(*args):
    print(type(args))  # 튜플
    print(args)

test(1, 2, 3)

In [ ]:
# *args: 묶인 것을 다시 풀어서 전달
args = (1, 2)
add(*args)  # add(1, 2) 와 동일

---

## Part 4. `**kwargs` — 키워드 인자 묶기/풀기

In [ ]:
# **kwargs: key=value 형태로 받기
def test(**kwargs):
    print(type(kwargs))  # 딕셔너리
    print(kwargs)

test(a=1, b=2)

In [ ]:
# **kwargs: 딕셔너리를 풀어서 전달
def test(a, b):
    print(a, b)

kwargs = {"a": 10, "b": 20}
test(**kwargs)  # test(a=10, b=20) 와 동일

---

## Part 5. 헷갈린 포인트 — 직접 확인하기

In [ ]:
# 호출 방식 vs 출력 결과 구분하기
def test(a, b):
    print(a, b)  # 출력은 print가 결정

kwargs = {"a": 10, "b": 20}

test(**kwargs)
# 흐름:
# test(**kwargs)
# → test(a=10, b=20)  ← 호출 방식
# → print(a, b)
# → 10 20             ← 출력 결과

In [ ]:
# ⚠️ *kwargs (별 하나) vs **kwargs (별 두 개)
def test(a, b):
    print(a, b)

kwargs = {"a": 100, "b": 200}

# 별 두 개: 값이 전달됨
test(**kwargs)  # → 100 200

In [ ]:
# 별 하나: 키(문자열)가 위치 인자로 전달됨
test(*kwargs)  # → a b (값이 아니라 키!)

| 문법 | 결과 |
|------|------|
| `*kwargs` | 키만 전달 (`"a"`, `"b"`) |
| `**kwargs` | key=value 전달 (`a=100`, `b=200`) |

---

## Part 6. 완성형 데코레이터 — 어떤 함수든 대응

In [ ]:
# 문제: wrapper()에 인자가 없으면 인자 있는 함수가 깨짐
def bad_decorator(func):
    def wrapper():
        func()  # 인자 전달 불가
    return wrapper

@bad_decorator
def add(a, b):
    return a + b

try:
    add(1, 2)
except TypeError as e:
    print(f"에러: {e}")

In [ ]:
# 해결: *args, **kwargs + return
def good_decorator(func):
    def wrapper(*args, **kwargs):      # 어떤 인자든 받음
        print(f"[실행] {func.__name__}")
        result = func(*args, **kwargs)  # 그대로 전달
        return result                   # 반드시 반환
    return wrapper

@good_decorator
def add(a, b):
    return a + b

@good_decorator
def greet(name, greeting="안녕"):
    return f"{greeting}, {name}!"

print(add(1, 2))
print(greet("철수"))
print(greet("영희", greeting="반가워"))

> **표준형 외우기**
> ```python
> def wrapper(*args, **kwargs):
>     result = func(*args, **kwargs)
>     return result
> ```
> `return`을 빠뜨리면 결과가 `None`으로 사라진다.

---

## Part 7. 등록형 데코레이터 — 자동화에서 가장 많이 쓰는 패턴

In [ ]:
# wrapper 없이 함수를 목록에만 등록하는 패턴
tools = []

def register(func):
    tools.append(func)  # 등록만 하고
    return func         # 함수 그대로 반환 (실행 변화 없음)

@register
def send_slack(message: str):
    """Slack으로 메시지를 보낸다"""
    print(f"Slack: {message}")

@register
def save_to_notion(title: str, content: str):
    """Notion에 페이지를 저장한다"""
    print(f"Notion: [{title}] {content}")

print("등록된 툴:", [f.__name__ for f in tools])

In [ ]:
# API 응답으로 함수 이름 + 인자가 오면 → 찾아서 실행
def run_tool(name, **kwargs):
    func = next(f for f in tools if f.__name__ == name)
    return func(**kwargs)

# Claude API 응답이 이런 식으로 온다고 가정:
# {"name": "send_slack", "arguments": {"message": "배포 완료!"}}

run_tool("send_slack", message="배포 완료!")
run_tool("save_to_notion", title="회의록", content="내일 미팅 확인")

| 데코레이터 종류 | 구조 | 용도 |
|----------------|------|------|
| wrapper형 | `func` → `wrapper` 반환 | 실행 전후 기능 추가 (로그, 타이머) |
| 등록형 | `func` 그대로 반환 | 함수 목록 관리 (AI 툴 등록, 라우터) |

---

## 최종 확인 문제

아래 코드 실행 전에 결과를 예측해봐.

In [ ]:
# Q1. 출력 결과는?
def test(a, b):
    print(a, b)

kwargs = {"a": 100, "b": 200}
test(*kwargs)  # 예측: ?

In [ ]:
# Q2. return이 없으면 어떻게 되나?
def decorator(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)  # return 없음
    return wrapper

@decorator
def add(a, b):
    return a + b

result = add(1, 2)
print(result)  # 예측: ?

In [ ]:
# Q3. 이 데코레이터는 wrapper형? 등록형?
tools = []

def tool(func):
    tools.append(func)
    return func

@tool
def hello():
    print("hello")

hello()  # 실행은 되나? 어떻게 출력?

---

## 핵심 요약

| 개념 | 핵심 |
|------|------|
| 함수는 변수 | `a = hello` → 담기 가능, `a()` → 실행 |
| 데코레이터 | 함수를 받아서 새 함수로 바꿔치기 |
| `*args` | 위치 인자를 튜플로 묶기/풀기 |
| `**kwargs` | 키워드 인자를 딕셔너리로 묶기/풀기 |
| `*kwargs` ❌ | 키 문자열만 전달됨 — 절대 쓰지 말 것 |
| `return` 필수 | 없으면 결과가 `None`으로 사라짐 |

> **한 줄 정리**
> 자동화 툴을 연결하는 것보다, 함수가 어떻게 전달되고 실행되는지를 이해할 때 비로소 내 것이 된다.